In [1]:
import pandas as pd
import numpy as np

In [2]:
admissions = pd.read_csv('HOSP/admissions_new.csv')
d_icd_diagnoses = pd.read_csv('HOSP/d_icd_diagnoses.csv')
diagnoses_icd = pd.read_csv('HOSP/diagnoses_icd.csv')
drgcodes = pd.read_csv('HOSP/drgcodes_new.csv')
omr = pd.read_csv('HOSP/omr_new.csv')
patients = pd.read_csv('HOSP/patients.csv')

icustays = pd.read_csv('ICU/icustays_new.csv')

triage = pd.read_csv('ED/triage_new.csv')

discharge = pd.read_csv('NOTES/discharge_new.csv')

In [3]:
# Turn dod to Bool type
patients['dod'] = patients['dod'].notnull().astype(int)

# Calculate the time of patients staying in the ICU
icustays['intime'] = pd.to_datetime(icustays['intime'])
icustays['outtime'] = pd.to_datetime(icustays['outtime'])
icustays['stay'] = (icustays['outtime'] - icustays['intime']).dt.total_seconds() / 3600

#### Choose the following variable to build a model.
triage['heartrate']
triage['temperature']
triage['resprate']
triage['o2sat']
triage['sbp']
triage['dbp']
triage['pain']
triage['acuity']
triage['chiefcomplaint']

icustays['stay']

patients['anchor_age']
patients['gender']
patients['dod']

drgcodes['drg_code']
drgcodes['drg_severity']
drgcodes['drg_mortality']

In [4]:
# Pivot the `omr` DataFrame to get the desired columns
omr_df = omr.pivot_table(index=['subject_id', 'hadm_id'], 
                         columns='result_name', 
                         values='result_value', 
                         aggfunc='first').reset_index()

# Select the required columns
omr_df = omr_df[['subject_id', 'hadm_id', 'BMI (kg/m2)', 'Height (Inches)', 'Weight (Lbs)', 'Blood Pressure']]

# Fill missing BMI values using the relationship: BMI = (Weight (Lbs) / (Height (Inches)^2)) * 703
omr_df['BMI (kg/m2)'] = omr_df['BMI (kg/m2)'].astype(float)
omr_df['Height (Inches)'] = omr_df['Height (Inches)'].astype(float)
omr_df['Weight (Lbs)'] = omr_df['Weight (Lbs)'].astype(float)

# Calculate BMI where it is missing and both Height and Weight are available
missing_bmi_mask = omr_df['BMI (kg/m2)'].isna() & omr_df['Height (Inches)'].notna() & omr_df['Weight (Lbs)'].notna()
omr_df.loc[missing_bmi_mask, 'BMI (kg/m2)'] = (omr_df['Weight (Lbs)'] / (omr_df['Height (Inches)'] ** 2)) * 703.06958

omr_df = omr_df.dropna(subset=['BMI (kg/m2)', 'Blood Pressure'])
omr_df = omr_df.drop(columns=['Height (Inches)', 'Weight (Lbs)'])

omr_df = omr_df.merge(patients[['subject_id', 'anchor_age', 'gender', 'dod']], on='subject_id', how='left')

triage_columns = ['hadm_id', 'heartrate', 'temperature', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity', 'chiefcomplaint']
omr_df = omr_df.merge(triage[triage_columns], on='hadm_id', how='left')

# 查找是否有重复的 hadm_id
duplicate_hadm_ids = omr_df[omr_df['hadm_id'].duplicated(keep=False)]
omr_df = omr_df.drop_duplicates(subset='hadm_id', keep=False)
omr_df = omr_df.merge(icustays[['hadm_id', 'stay']], on='hadm_id', how='left')

# Find rows with duplicate `hadm_id`
duplicate_hadm_ids = omr_df[omr_df['hadm_id'].duplicated(keep=False)]

# Display the duplicate rows
# Sum the 'stay' values for duplicate `hadm_id`
duplicate_hadm_ids = duplicate_hadm_ids.groupby('hadm_id', as_index=False).agg({'stay': 'sum'})

# Display the updated DataFrame
duplicate_hadm_ids
# Update the `stay` values in `omr_df` using the summed `stay` values from `duplicate_hadm_ids`
omr_df = omr_df.merge(duplicate_hadm_ids, on='hadm_id', how='left', suffixes=('', '_new'))

# Replace the `stay` column with the new values where available
omr_df['stay'] = omr_df['stay_new'].fillna(omr_df['stay'])

# Drop the temporary `stay_new` column
omr_df = omr_df.drop(columns=['stay_new'])

# Ensure no duplicate `hadm_id` rows remain
omr_df = omr_df.drop_duplicates(subset='hadm_id', keep='first')

# 选择需要的列
drg_columns = ['hadm_id', 'drg_code', 'drg_severity', 'drg_mortality']

# 将 drgcodes 数据合并到 omr_df
omr_df = omr_df.merge(drgcodes[drg_columns], on='hadm_id', how='left')

omr_df['stay'] = omr_df['stay'].fillna(0)

# Add text_vector to omr_df by matching hadm_id
# omr_df['text_vector'] = omr_df['hadm_id'].map(hadm_id_to_vector)

# Split the 'Blood Pressure' column into 'Blood Pressure_h' and 'Blood Pressure_l'
omr_df[['Blood Pressure_h', 'Blood Pressure_l']] = omr_df['Blood Pressure'].str.split('/', expand=True).astype(float)
omr_df = omr_df.drop(columns=['Blood Pressure'])

omr_df['gender'] = omr_df['gender'].map({'M': 0, 'F': 1})

omr_df['pain'] = pd.to_numeric(omr_df['pain'], errors='coerce')
omr_df = omr_df.dropna()

# Display the resulting DataFrame
omr_df

,subject_id,hadm_id,BMI (kg/m2),anchor_age,gender,dod,heartrate,temperature,resprate,o2sat,...,dbp,pain,acuity,chiefcomplaint,stay,drg_code,drg_severity,drg_mortality,Blood Pressure_h,Blood Pressure_l
0,10000980,24947999.0,32.3,73,1,1,88.0,98.0,18.0,98.0,...,120.0,0.0,2.0,Dyspnea,0.0,194,2.0,3.0,186.0,79.0
1,10001884,29678536.0,25.5,68,1,1,110.0,98.5,26.0,92.0,...,85.0,0.0,2.0,"Dyspnea, Atrial fibrillation",0.0,201,2.0,1.0,120.0,75.0
3,10002221,21008195.0,36.3,68,1,0,82.0,98.4,18.0,95.0,...,70.0,0.0,2.0,Chest pain,0.0,134,2.0,1.0,116.0,56.0
6,10003019,21223482.0,25.2,69,0,0,84.0,97.9,18.0,100.0,...,70.0,2.0,2.0,Syncope,0.0,204,2.0,2.0,110.0,68.0
9,10007795,22051341.0,23.7,53,1,0,108.0,98.6,18.0,95.0,...,61.0,7.0,3.0,"Abd pain, Back pain",0.0,422,1.0,1.0,147.0,97.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12273,19992875,29951097.0,25.9,37,0,0,72.0,96.6,18.0,100.0,...,88.0,7.0,2.0,"Transfer, GI bleed",0.0,253,3.0,2.0,118.0,77.0
12275,19997367,23087270.0,27.4,63,1,0,79.0,98.5,18.0,100.0,...,69.0,0.0,3.0,ALLERGIC REACTION,0.0,721,3.0,2.0,110.0,57.0
12277,19997911,25785472.0,27.1,79,1,0,50.0,98.3,16.0,96.0,...,54.0,0.0,3.0,Abnormal sodium level,0.0,424,2.0,2.0,194.0,75.0
12279,19998562,26846592.0,21.4,79,0,1,64.0,98.0,18.0,99.0,...,56.0,0.0,2.0,Syncope,0.0,424,3.0,2.0,113.0,52.0


In [5]:
# Select rows without missing values
omr_df_n = omr_df.dropna()

# Display the resulting DataFrame
omr_df_n

,subject_id,hadm_id,BMI (kg/m2),anchor_age,gender,dod,heartrate,temperature,resprate,o2sat,...,dbp,pain,acuity,chiefcomplaint,stay,drg_code,drg_severity,drg_mortality,Blood Pressure_h,Blood Pressure_l
0,10000980,24947999.0,32.3,73,1,1,88.0,98.0,18.0,98.0,...,120.0,0.0,2.0,Dyspnea,0.0,194,2.0,3.0,186.0,79.0
1,10001884,29678536.0,25.5,68,1,1,110.0,98.5,26.0,92.0,...,85.0,0.0,2.0,"Dyspnea, Atrial fibrillation",0.0,201,2.0,1.0,120.0,75.0
3,10002221,21008195.0,36.3,68,1,0,82.0,98.4,18.0,95.0,...,70.0,0.0,2.0,Chest pain,0.0,134,2.0,1.0,116.0,56.0
6,10003019,21223482.0,25.2,69,0,0,84.0,97.9,18.0,100.0,...,70.0,2.0,2.0,Syncope,0.0,204,2.0,2.0,110.0,68.0
9,10007795,22051341.0,23.7,53,1,0,108.0,98.6,18.0,95.0,...,61.0,7.0,3.0,"Abd pain, Back pain",0.0,422,1.0,1.0,147.0,97.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12273,19992875,29951097.0,25.9,37,0,0,72.0,96.6,18.0,100.0,...,88.0,7.0,2.0,"Transfer, GI bleed",0.0,253,3.0,2.0,118.0,77.0
12275,19997367,23087270.0,27.4,63,1,0,79.0,98.5,18.0,100.0,...,69.0,0.0,3.0,ALLERGIC REACTION,0.0,721,3.0,2.0,110.0,57.0
12277,19997911,25785472.0,27.1,79,1,0,50.0,98.3,16.0,96.0,...,54.0,0.0,3.0,Abnormal sodium level,0.0,424,2.0,2.0,194.0,75.0
12279,19998562,26846592.0,21.4,79,0,1,64.0,98.0,18.0,99.0,...,56.0,0.0,2.0,Syncope,0.0,424,3.0,2.0,113.0,52.0


In [14]:
fjy = pd.read_csv('yhy.csv')
yhy = pd.read_csv('hadm_id_selected.csv')

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from scipy.sparse import hstack
from scipy.sparse import vstack
from sklearn.metrics import roc_auc_score

In [15]:
fjy_texts = discharge[discharge['hadm_id'].isin(fjy['hadm_id'])]
fjy_texts

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
111132,14281506-DS-33,14281506,24243183,DS,33,2167-08-23 00:00:00,2167-08-23 20:22:00,\nName: ___ Unit No: ___\n \...
111208,14283639-DS-9,14283639,28170780,DS,9,2164-04-16 00:00:00,2164-04-17 16:06:00,\nName: ___ Unit No: ...
111240,14285792-DS-16,14285792,23114985,DS,16,2114-02-03 00:00:00,2114-02-03 17:28:00,\nName: ___ Unit No: ___\n \n...
111279,14286519-DS-16,14286519,24567679,DS,16,2203-02-02 00:00:00,2203-02-03 19:39:00,\nName: ___ Unit No: ___\...
111369,14290075-DS-17,14290075,21654048,DS,17,2167-06-26 00:00:00,2167-07-03 15:37:00,\nName: ___ Unit No: ___\...
...,...,...,...,...,...,...,...,...
259192,19981610-DS-10,19981610,29416526,DS,10,2177-01-31 00:00:00,2177-02-02 15:47:00,\nName: ___. Unit No: ___\...
259216,19983009-DS-20,19983009,26466419,DS,20,2142-09-09 00:00:00,2142-09-09 15:03:00,\nName: ___ Unit No: __...
259426,19990141-DS-18,19990141,24852269,DS,18,2133-03-05 00:00:00,2133-03-05 21:55:00,\nName: ___ Unit No: ___\n ...
259489,19992875-DS-34,19992875,29951097,DS,34,2163-04-03 00:00:00,2163-04-03 17:04:00,\nName: ___. Unit No: ___\n \...


In [12]:
filtered_texts = discharge[discharge['hadm_id'].isin(omr_df['hadm_id'])]
filtered_texts

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
16,10000980-DS-22,10000980,24947999,DS,22,2190-11-08 00:00:00,2190-11-09 13:57:00,\nName: ___ Unit No: ___\n \nAdmi...
40,10001884-DS-32,10001884,29678536,DS,32,2130-10-12 00:00:00,2130-10-13 22:19:00,\nName: ___ Unit No: ___\n \nA...
64,10002221-DS-8,10002221,21008195,DS,8,2200-10-01 00:00:00,2200-10-01 15:54:00,\nName: ___ Unit No: ___\n ...
97,10003019-DS-23,10003019,21223482,DS,23,2175-11-02 00:00:00,2175-11-02 22:33:00,\nName: ___. Unit No: ___\n \...
220,10007795-DS-17,10007795,22051341,DS,17,2136-09-24 00:00:00,2136-09-24 11:54:00,\nName: ___ Unit No: ...
...,...,...,...,...,...,...,...,...
259489,19992875-DS-34,19992875,29951097,DS,34,2163-04-03 00:00:00,2163-04-03 17:04:00,\nName: ___. Unit No: ___\n \...
259597,19997367-DS-16,19997367,23087270,DS,16,2126-02-01 00:00:00,2126-02-02 17:03:00,\nName: ___ Unit No: ___\...
259617,19997911-DS-22,19997911,25785472,DS,22,2199-11-12 00:00:00,2199-11-12 16:52:00,\nName: ___ Unit No: ___\...
259635,19998562-DS-21,19998562,26846592,DS,21,2166-04-16 00:00:00,2166-04-17 16:08:00,\nName: ___ Unit No: ___\...


#### Try DistilBERT
DistilClinicalBERT is a distilled version of the BioClinicalBERT model which is distilled for 3 epochs using a total batch size of 192 on the MIMIC-III notes dataset.

In [9]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import gc 
from torch.optim import AdamW
from tqdm import tqdm

c:\Users\tonglingwang\.conda\envs\torch_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Load DistilClinicalBERT tokenizer and model
from src.features.text_embedding import (
    tokenize_and_block,
    add_special_tokens_to_blocks,
    extract_block_embeddings as extract_features_from_blocks,
    reduce_block_embeddings_to_document_vector as reduce_to_768,
)
from src.features.engineering import parse_vector_column, build_sparse_matrix_from_vector_column
tokenizer = AutoTokenizer.from_pretrained("nlpie/distil-clinicalbert")
model = AutoModel.from_pretrained("nlpie/distil-clinicalbert")

Some weights of BertModel were not initialized from the model checkpoint at nlpie/distil-clinicalbert and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
# Define the block size (510 tokens for each block, accounting for CLS and SEP tokens)
block_size = 510
special_tokens = tokenizer(["[CLS]", "[SEP]", "[PAD]"], is_split_into_words=True)
# tokenize_and_block, add_special_tokens_to_blocks, extract_features_from_blocks, reduce_to_768: see src.features.text_embedding

In [17]:
# reduce_to_768 is imported as reduce_block_embeddings_to_document_vector from src.features.text_embedding (see load cell above)

In [21]:
filt_2000 = filtered_texts.sample(n = 2000, random_state=5230)

In [93]:
# Example usage:
corpus = filt_2000['text'].tolist()  # Replace with your actual text corpus
all_features = []
# Open a file to save features and corresponding hadm_id
with open("fid_seed.txt", "a") as file:
        for i, text in enumerate(corpus):
                # Step 1: Tokenizing & Blocking
                blocks = tokenize_and_block(text, block_size)
                
                # Step 2: Add special tokens and padding
                blocks = add_special_tokens_to_blocks(blocks)
                
                # Step 3: Feature extraction (extract embeddings)
                features = extract_features_from_blocks(blocks, model, tokenizer)
                # Reduce the feature's dimension
                features = reduce_to_768(features)
                
                # Save the features and hadm_id
                hadm_id = filtered_texts.iloc[i]['hadm_id']
                file.write(f"{hadm_id}\t{features.tolist()}\n")

                # Release memory
                del blocks, features
                gc.collect()

Token indices sequence length is longer than the specified maximum sequence length for this model (3642 > 512). Running this sequence through the model will result in indexing errors


add fjy's examples in

In [20]:
# Example usage:
corpus = fjy_texts['text'].tolist()  # Replace with your actual text corpus
all_features = []
# Open a file to save features and corresponding hadm_id
with open("fid_seed2.txt", "a") as file:
        for i, text in enumerate(corpus):
                # Step 1: Tokenizing & Blocking
                blocks = tokenize_and_block(text, block_size)
                
                # Step 2: Add special tokens and padding
                blocks = add_special_tokens_to_blocks(blocks)
                
                # Step 3: Feature extraction (extract embeddings)
                features = extract_features_from_blocks(blocks, model, tokenizer)
                # Reduce the feature's dimension
                features = reduce_to_768(features)
                
                # Save the features and hadm_id
                hadm_id = fjy_texts.iloc[i]['hadm_id']
                file.write(f"{hadm_id}\t{features.tolist()}\n")

                # Release memory
                del blocks, features
                gc.collect()

In [203]:
import pickle

# Example usage:
corpus = filt_2000['text'].tolist()  # Replace with your actual text corpus
all_features = []

# Iterate through the corpus and process each text
for i, text in enumerate(corpus):
    # Step 1: Tokenizing & Blocking
    blocks = tokenize_and_block(text, block_size)
    
    # Step 2: Add special tokens and padding
    blocks = add_special_tokens_to_blocks(blocks)
    
    # Step 3: Feature extraction (extract embeddings)
    features = extract_features_from_blocks(blocks, model, tokenizer)
    
    # Reduce the feature's dimension
    features = reduce_to_768(features)
    
    # Save the features and hadm_id
    hadm_id = filtered_texts.iloc[i]['hadm_id']
    all_features.append({'hadm_id': hadm_id, 'features': features})

    # Save all features and hadm_id to a single .pkl file
    with open("fid_seed.pkl", "wb") as pkl_file:
        pickle.dump(all_features, pkl_file)

    # Release memory
    del blocks, features
    gc.collect()

In [ ]:
# Define optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Fine-tune the model on your dataset
model.train()

for epoch in range(num_epochs):
    for batch in dataloader:  # Replace with your DataLoader
        inputs = tokenizer(batch['text'], padding=True, truncation=True, return_tensors="pt")
        labels = batch['labels']  # Replace with your labels
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

print("Training completed.")

NameError: name 'num_epochs' is not defined

##### Data Processing befor Regression

In [32]:
df2 = pd.read_csv('fid_seed2.txt', sep="\t", header=None, names=["hadm_id", "text_vector"])
df1 = pd.read_csv('fid_seed.txt', sep="\t", header=None, names=["hadm_id", "text_vector"])
combined_df1 = omr_df_n.merge(pd.concat([df1, df2]), on='hadm_id', how='inner')

In [34]:
combined_df1 = parse_vector_column(combined_df1, "text_vector")
combined_df1['drg_code'] = combined_df1['drg_code'].astype('category')

# Save combined_df1 to a pickle file
combined_df1.to_pickle('combined_df1.pkl')

In [158]:
df1 = pd.read_csv('fid_seed.txt', sep="\t", header=None, names=["hadm_id", "text_vector"])

# Merge df1 and omr_df_n on 'hadm_id'
combined_df = omr_df_n.merge(df1, on='hadm_id', how='inner')
combined_df = parse_vector_column(combined_df, "text_vector")
combined_df['drg_code'] = combined_df['drg_code'].astype('category')

# Save combined_df to a CSV file
combined_df.to_csv('combined_df.csv', index=False)

In [37]:
combined_df = pd.read_pickle('combined_df1.pkl')

In [38]:
combined_df

,subject_id,hadm_id,BMI (kg/m2),anchor_age,gender,dod,heartrate,temperature,resprate,o2sat,...,pain,acuity,chiefcomplaint,stay,drg_code,drg_severity,drg_mortality,Blood Pressure_h,Blood Pressure_l,text_vector
0,10000980,24947999.0,32.3,73,1,1,88.0,98.0,18.0,98.0,...,0.0,2.0,Dyspnea,0.0,194,2.0,3.0,186.0,79.0,"[0.031223779544234276, -0.07917327433824539, -..."
1,10001884,29678536.0,25.5,68,1,1,110.0,98.5,26.0,92.0,...,0.0,2.0,"Dyspnea, Atrial fibrillation",0.0,201,2.0,1.0,120.0,75.0,"[0.08313854783773422, -0.11806894838809967, -0..."
2,10002221,21008195.0,36.3,68,1,0,82.0,98.4,18.0,95.0,...,0.0,2.0,Chest pain,0.0,134,2.0,1.0,116.0,56.0,"[0.012135172262787819, -0.11237561702728271, 0..."
3,10003019,21223482.0,25.2,69,0,0,84.0,97.9,18.0,100.0,...,2.0,2.0,Syncope,0.0,204,2.0,2.0,110.0,68.0,"[0.05772009491920471, -0.06333684176206589, 0...."
4,10007795,22051341.0,23.7,53,1,0,108.0,98.6,18.0,95.0,...,7.0,3.0,"Abd pain, Back pain",0.0,422,1.0,1.0,147.0,97.0,"[0.06341613084077835, -0.09620142728090286, 0...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3163,19981610,29416526.0,34.0,21,1,0,79.0,98.0,16.0,99.0,...,0.0,3.0,HYPOCALCEMIA,0.0,425,1.0,1.0,110.0,64.0,"[0.030010048300027847, -0.1173621267080307, -0..."
3164,19983009,26466419.0,19.1,56,0,1,66.0,98.2,18.0,100.0,...,0.0,2.0,Hypokalemia,0.0,425,3.0,3.0,117.0,70.0,"[0.06512671709060669, -0.09365414828062057, -0..."
3165,19990141,24852269.0,25.8,79,0,0,67.0,97.6,18.0,100.0,...,5.0,3.0,"DVT, Chest pain, R Leg pain",0.0,134,1.0,1.0,102.0,64.0,"[0.06681495159864426, -0.11060956865549088, -0..."
3166,19992875,29951097.0,25.9,37,0,0,72.0,96.6,18.0,100.0,...,7.0,2.0,"Transfer, GI bleed",0.0,253,3.0,2.0,118.0,77.0,"[0.05936222895979881, -0.12810805439949036, -0..."


In [39]:
# Load model and tokenizer
model_name = "nlpie/distil-clinicalbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Function to get embedding from text
def encode_text(text, tokenizer, model, device):
    with torch.no_grad():
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
        # Use CLS token representation
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # shape: (1, 768)
        return cls_embedding.squeeze().cpu().numpy()

# Encode all chiefcomplaints
encoded_chiefcomplaints = []
for text in tqdm(combined_df['chiefcomplaint'], desc="Encoding chief complaints"):
    embedding = encode_text(str(text), tokenizer, model, device)
    encoded_chiefcomplaints.append(embedding)

# Add the embedding as a new column (optional: convert to array or sparse matrix later)
combined_df['chiefcomplaint_vector'] = encoded_chiefcomplaints
combined_df['chiefcomplaint_vector'] = combined_df['chiefcomplaint_vector'].apply(lambda x: np.array(x))

Some weights of BertModel were not initialized from the model checkpoint at nlpie/distil-clinicalbert and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Encoding chief complaints: 100%|██████████| 3168/3168 [00:48<00:00, 65.11it/s]


In [40]:
combined_df.to_pickle("combined_df_with_chiefcomplaint1.pkl")

In [45]:
combined_df = pd.read_pickle("combined_df_with_chiefcomplaint1.pkl")
combined_df

,subject_id,hadm_id,BMI (kg/m2),anchor_age,gender,dod,heartrate,temperature,resprate,o2sat,...,acuity,chiefcomplaint,stay,drg_code,drg_severity,drg_mortality,Blood Pressure_h,Blood Pressure_l,text_vector,chiefcomplaint_vector
0,10000980,24947999.0,32.3,73,1,1,88.0,98.0,18.0,98.0,...,2.0,Dyspnea,0.0,194,2.0,3.0,186.0,79.0,"[0.031223779544234276, -0.07917327433824539, -...","[0.064169675, 0.18310219, -0.52304167, 0.86679..."
1,10001884,29678536.0,25.5,68,1,1,110.0,98.5,26.0,92.0,...,2.0,"Dyspnea, Atrial fibrillation",0.0,201,2.0,1.0,120.0,75.0,"[0.08313854783773422, -0.11806894838809967, -0...","[-0.16537252, 0.25593957, -0.28182814, 0.71011..."
2,10002221,21008195.0,36.3,68,1,0,82.0,98.4,18.0,95.0,...,2.0,Chest pain,0.0,134,2.0,1.0,116.0,56.0,"[0.012135172262787819, -0.11237561702728271, 0...","[0.34217817, -0.014103327, -0.24728408, 0.7728..."
3,10003019,21223482.0,25.2,69,0,0,84.0,97.9,18.0,100.0,...,2.0,Syncope,0.0,204,2.0,2.0,110.0,68.0,"[0.05772009491920471, -0.06333684176206589, 0....","[0.36482033, 0.4075926, -0.3328574, 0.47741792..."
4,10007795,22051341.0,23.7,53,1,0,108.0,98.6,18.0,95.0,...,3.0,"Abd pain, Back pain",0.0,422,1.0,1.0,147.0,97.0,"[0.06341613084077835, -0.09620142728090286, 0....","[0.3460847, -0.07331242, -0.35644135, 0.922641..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3163,19981610,29416526.0,34.0,21,1,0,79.0,98.0,16.0,99.0,...,3.0,HYPOCALCEMIA,0.0,425,1.0,1.0,110.0,64.0,"[0.030010048300027847, -0.1173621267080307, -0...","[0.15528555, -0.37537834, -0.03744465, 0.54269..."
3164,19983009,26466419.0,19.1,56,0,1,66.0,98.2,18.0,100.0,...,2.0,Hypokalemia,0.0,425,3.0,3.0,117.0,70.0,"[0.06512671709060669, -0.09365414828062057, -0...","[0.004019415, -0.086740956, 0.05278576, 0.7256..."
3165,19990141,24852269.0,25.8,79,0,0,67.0,97.6,18.0,100.0,...,3.0,"DVT, Chest pain, R Leg pain",0.0,134,1.0,1.0,102.0,64.0,"[0.06681495159864426, -0.11060956865549088, -0...","[0.44183815, -0.14860332, -0.14596382, 0.75898..."
3166,19992875,29951097.0,25.9,37,0,0,72.0,96.6,18.0,100.0,...,2.0,"Transfer, GI bleed",0.0,253,3.0,2.0,118.0,77.0,"[0.05936222895979881, -0.12810805439949036, -0...","[0.5521446, 0.034858927, 0.0056764483, 0.53335..."


#####  Try Logistic Regression
use combined_df to do LogisticRegression, let combined_df['dod'] as response variable, and all other variables(except subject_id and hadm_id) as predictor variable. Noted, you need to change the types of all variables to be able to fit into the model.

In [43]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from scipy.sparse import csr_matrix, hstack, vstack
from sklearn.preprocessing import StandardScaler

In [131]:
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# Prepare the numeric features
X_numeric = combined_df[['BMI (kg/m2)', 'anchor_age', 'gender', 'stay', 'heartrate', 'temperature', 'resprate',
                         'o2sat', 'sbp', 'dbp', 'drg_code', 'drg_severity', 'drg_mortality', 
                         'Blood Pressure_h', 'Blood Pressure_l']].values

# Convert text_vector list into a sparse matrix (each row is a vector)
X_text_sparse = build_sparse_matrix_from_vector_column(combined_df, "text_vector")
# Convert chiefcomplaint_vector list into a sparse matrix (each row is a vector)
X_chiefcomplaint_sparse = build_sparse_matrix_from_vector_column(combined_df, "chiefcomplaint_vector")

# Combine numerical, text, and chiefcomplaint features
X = hstack([csr_matrix(X_numeric), X_text_sparse, X_chiefcomplaint_sparse])

# Target variable
y = combined_df['dod'].astype(int)  # Make sure y is numeric

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5230)

# Train logistic regression
model = LogisticRegression(max_iter=100, class_weight='balanced', warm_start=True, tol=0.1, C=0.1)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# AU-ROC
roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("AU-ROC Score:", roc_auc)


Accuracy: 0.6876971608832808

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.70      0.72       366
           1       0.62      0.67      0.64       268

    accuracy                           0.69       634
   macro avg       0.68      0.69      0.68       634
weighted avg       0.69      0.69      0.69       634

AU-ROC Score: 0.7341570834352825


c:\Users\tonglingwang\.conda\envs\torch_env\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


#### LightGBM

In [152]:
import lightgbm as lgb

# Prepare numeric features
X_numeric = combined_df[['BMI (kg/m2)', 'anchor_age', 'gender', 'stay', 'heartrate', 'temperature', 'resprate',
                         'o2sat', 'sbp', 'dbp', 'drg_code', 'drg_severity', 'drg_mortality', 
                         'Blood Pressure_h', 'Blood Pressure_l']].values

# Convert text_vector list into sparse matrix
X_text_sparse = build_sparse_matrix_from_vector_column(combined_df, "text_vector")

# Convert chiefcomplaint_vector list into a sparse matrix (each row is a vector)
X_chiefcomplaint_sparse = build_sparse_matrix_from_vector_column(combined_df, "chiefcomplaint_vector")

# Combine numerical, text, and chiefcomplaint features
X = hstack([csr_matrix(X_numeric), X_text_sparse, X_chiefcomplaint_sparse])
y = combined_df['dod'].astype(int)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5230)

# Create LightGBM Dataset objects
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test)

# Set parameters for LightGBM
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'seed': 42,
    'verbosity': -1
}

# Initialize LightGBM
lgb_model = lgb.LGBMClassifier(class_weight="balanced")

# Train the model with early stopping
lgb_model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, test_data],  # Validation data for early stopping
    num_boost_round=1000,  # Maximum number of iterations
  #  early_stopping_rounds=50,  # Stop after 50 rounds without improvement
  #  verbose_eval=100  # Print progress every 100 iterations
)

# Predict using the best iteration
y_pred_prob = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)
y_pred = (y_pred_prob >= 0.08).astype(int)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("AU-ROC Score:", roc_auc_score(y_test, y_pred_prob))


Accuracy: 0.6514195583596214

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.60      0.67       366
           1       0.57      0.72      0.63       268

    accuracy                           0.65       634
   macro avg       0.66      0.66      0.65       634
weighted avg       0.67      0.65      0.65       634

AU-ROC Score: 0.7111879128945438


In [52]:
import lightgbm as lgb

# Prepare numeric features
X_numeric = combined_df[['BMI (kg/m2)', 'anchor_age', 'gender', 'stay', 'heartrate', 'temperature', 'resprate',
                         'o2sat', 'sbp', 'dbp', 'drg_code', 'drg_severity', 'drg_mortality', 
                         'Blood Pressure_h', 'Blood Pressure_l']].values

# Convert text_vector list into sparse matrix
X_text_sparse = build_sparse_matrix_from_vector_column(combined_df, "text_vector")

# Combine numeric and text features
X = hstack([csr_matrix(X_numeric), X_text_sparse])
y = combined_df['dod'].astype(int)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5230)

# Create LightGBM Dataset objects
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test)

# Set parameters for LightGBM
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'seed': 42,
    'verbosity': -1
}

# Initialize LightGBM
lgb_model = lgb.LGBMClassifier(class_weight="balanced")

# Train the model with early stopping
lgb_model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, test_data],  # Validation data for early stopping
    num_boost_round=1000,  # Maximum number of iterations
  #  early_stopping_rounds=50,  # Stop after 50 rounds without improvement
  #  verbose_eval=100  # Print progress every 100 iterations
)

# Predict using the best iteration
y_pred_prob = lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration)
y_pred = (y_pred_prob >= 0.5).astype(int)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("AU-ROC Score:", roc_auc_score(y_test, y_pred_prob))


Accuracy: 0.6214511041009464

Classification Report:
               precision    recall  f1-score   support

           0       0.65      0.73      0.69       366
           1       0.56      0.47      0.51       268

    accuracy                           0.62       634
   macro avg       0.61      0.60      0.60       634
weighted avg       0.61      0.62      0.61       634

AU-ROC Score: 0.6816634042900253


#### RandomForestClassifier

In [70]:
from sklearn.ensemble import RandomForestClassifier

# Prepare numeric features
X_numeric = combined_df[['BMI (kg/m2)', 'anchor_age', 'gender', 'stay', 'heartrate', 'temperature', 'resprate',
                         'o2sat', 'sbp', 'dbp', 'drg_code', 'drg_severity', 'drg_mortality', 
                         'Blood Pressure_h', 'Blood Pressure_l']]

# Convert text_vector list into sparse matrix
X_text_sparse = build_sparse_matrix_from_vector_column(combined_df, "text_vector")

# Convert chiefcomplaint_vector list into a sparse matrix (each row is a vector)
X_chiefcomplaint_sparse = build_sparse_matrix_from_vector_column(combined_df, "chiefcomplaint_vector")

# Combine numerical, text, and chiefcomplaint features
X = hstack([csr_matrix(X_numeric), X_text_sparse, X_chiefcomplaint_sparse])
y = combined_df['dod'].astype(int)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5230)

# Initialize the model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=2,
    class_weight='balanced',
    random_state=5230,
    n_jobs=-1  # Use all available cores
)

# Train the model
rf_model.fit(X_train, y_train)

# Predict
y_pred = rf_model.predict(X_test)
y_pred_prob = rf_model.predict_proba(X_test)[:, 1]

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("AU-ROC Score:", roc_auc_score(y_test, y_pred_prob))


Accuracy: 0.6735015772870663

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.67      0.70       366
           1       0.60      0.67      0.63       268

    accuracy                           0.67       634
   macro avg       0.67      0.67      0.67       634
weighted avg       0.68      0.67      0.68       634

AU-ROC Score: 0.7110553788434876


In [74]:
df = pd.read_pickle("combined_df_with_chiefcomplaint1.pkl")

In [76]:
df['anchor_age'] = df['anchor_age'].astype('float64')
df['pain'] = df['pain'].astype('category')
df['acuity'] = df['acuity'].astype('int')
df['drg_severity'] = df['drg_severity'].astype('int')

from scipy.stats import zscore

# Select numerical columns excluding 'subject_id' and 'hadm_id'
numerical_cols = df.select_dtypes(include=['float64']).columns.difference(['subject_id', 'hadm_id'])

# Apply z-score normalization to numerical columns
df_normalized = df.copy()
df_normalized[numerical_cols] = df[numerical_cols].apply(zscore)

# Retain non-numerical columns as they are
df_normalized

,subject_id,hadm_id,BMI (kg/m2),anchor_age,gender,dod,heartrate,temperature,resprate,o2sat,...,acuity,chiefcomplaint,stay,drg_code,drg_severity,drg_mortality,Blood Pressure_h,Blood Pressure_l,text_vector,chiefcomplaint_vector
0,10000980,24947999.0,0.083212,0.770371,1,1,0.104498,-0.116510,-0.025431,0.000616,...,2,Dyspnea,-0.196569,194,2,0.839626,2.620476,0.487229,"[0.031223779544234276, -0.07917327433824539, -...","[0.064169675, 0.18310219, -0.52304167, 0.86679..."
1,10001884,29678536.0,-0.103756,0.446065,1,1,1.226385,0.188670,0.223710,-1.951563,...,2,"Dyspnea, Atrial fibrillation",-0.196569,201,2,-1.384400,-0.273329,0.217190,"[0.08313854783773422, -0.11806894838809967, -0...","[-0.16537252, 0.25593957, -0.28182814, 0.71011..."
2,10002221,21008195.0,0.193193,0.446065,1,0,-0.201472,0.127634,-0.025431,-0.975473,...,2,Chest pain,-0.196569,134,2,-1.384400,-0.448711,-1.065494,"[0.012135172262787819, -0.11237561702728271, 0...","[0.34217817, -0.014103327, -0.24728408, 0.7728..."
3,10003019,21223482.0,-0.112004,0.510926,0,0,-0.099482,-0.177546,-0.025431,0.651342,...,2,Syncope,-0.196569,204,2,-0.272387,-0.711784,-0.255378,"[0.05772009491920471, -0.06333684176206589, 0....","[0.36482033, 0.4075926, -0.3328574, 0.47741792..."
4,10007795,22051341.0,-0.153247,-0.526855,1,0,1.124395,0.249706,-0.025431,-0.975473,...,3,"Abd pain, Back pain",-0.196569,422,1,-1.384400,0.910501,1.702404,"[0.06341613084077835, -0.09620142728090286, 0....","[0.3460847, -0.07331242, -0.35644135, 0.922641..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3163,19981610,29416526.0,0.129954,-2.602416,1,0,-0.354456,-0.116510,-0.087716,0.325979,...,3,HYPOCALCEMIA,-0.196569,425,1,-1.384400,-0.711784,-0.525417,"[0.030010048300027847, -0.1173621267080307, -0...","[0.15528555, -0.37537834, -0.03744465, 0.54269..."
3164,19983009,26466419.0,-0.279725,-0.332271,0,1,-1.017390,0.005562,-0.025431,0.651342,...,2,Hypokalemia,-0.196569,425,3,0.839626,-0.404865,-0.120358,"[0.06512671709060669, -0.09365414828062057, -0...","[0.004019415, -0.086740956, 0.05278576, 0.7256..."
3165,19990141,24852269.0,-0.095507,1.159539,0,0,-0.966395,-0.360654,-0.025431,0.651342,...,3,"DVT, Chest pain, R Leg pain",-0.196569,134,1,-1.384400,-1.062548,-0.525417,"[0.06681495159864426, -0.11060956865549088, -0...","[0.44183815, -0.14860332, -0.14596382, 0.75898..."
3166,19992875,29951097.0,-0.092758,-1.564635,0,0,-0.711421,-0.971014,-0.025431,0.651342,...,2,"Transfer, GI bleed",-0.196569,253,3,-0.272387,-0.361020,0.352210,"[0.05936222895979881, -0.12810805439949036, -0...","[0.5521446, 0.034858927, 0.0056764483, 0.53335..."


In [77]:
# Apply average pooling to reduce 'chiefcomplaint_vector' to 1 dimension
df_normalized['chief_reduced'] = np.mean(np.vstack(df_normalized['chiefcomplaint_vector'].values), axis=1)

# Apply average pooling to reduce 'text_vector' to 3 dimensions
text_vectors = np.vstack(df_normalized['text_vector'].values)
df_normalized['text_dim1'] = np.mean(text_vectors[:, :text_vectors.shape[1] // 3], axis=1)
df_normalized['text_dim2'] = np.mean(text_vectors[:, text_vectors.shape[1] // 3:2 * text_vectors.shape[1] // 3], axis=1)
df_normalized['text_dim3'] = np.mean(text_vectors[:, 2 * text_vectors.shape[1] // 3:], axis=1)

# Perform z-score normalization on the 4 new vectors
df_normalized['chief_reduced'] = zscore(df_normalized['chief_reduced'])
df_normalized['text_dim1'] = zscore(df_normalized['text_dim1'])
df_normalized['text_dim2'] = zscore(df_normalized['text_dim2'])
df_normalized['text_dim3'] = zscore(df_normalized['text_dim3'])

In [78]:
df_normalized

,subject_id,hadm_id,BMI (kg/m2),anchor_age,gender,dod,heartrate,temperature,resprate,o2sat,...,drg_severity,drg_mortality,Blood Pressure_h,Blood Pressure_l,text_vector,chiefcomplaint_vector,chief_reduced,text_dim1,text_dim2,text_dim3
0,10000980,24947999.0,0.083212,0.770371,1,1,0.104498,-0.116510,-0.025431,0.000616,...,2,0.839626,2.620476,0.487229,"[0.031223779544234276, -0.07917327433824539, -...","[0.064169675, 0.18310219, -0.52304167, 0.86679...",-0.567731,0.682846,-1.284469,0.724215
1,10001884,29678536.0,-0.103756,0.446065,1,1,1.226385,0.188670,0.223710,-1.951563,...,2,-1.384400,-0.273329,0.217190,"[0.08313854783773422, -0.11806894838809967, -0...","[-0.16537252, 0.25593957, -0.28182814, 0.71011...",-0.024808,-0.047710,0.310135,-0.269413
2,10002221,21008195.0,0.193193,0.446065,1,0,-0.201472,0.127634,-0.025431,-0.975473,...,2,-1.384400,-0.448711,-1.065494,"[0.012135172262787819, -0.11237561702728271, 0...","[0.34217817, -0.014103327, -0.24728408, 0.7728...",0.969585,0.804683,0.906756,-1.546128
3,10003019,21223482.0,-0.112004,0.510926,0,0,-0.099482,-0.177546,-0.025431,0.651342,...,2,-0.272387,-0.711784,-0.255378,"[0.05772009491920471, -0.06333684176206589, 0....","[0.36482033, 0.4075926, -0.3328574, 0.47741792...",0.103533,-1.536779,0.832074,0.350447
4,10007795,22051341.0,-0.153247,-0.526855,1,0,1.124395,0.249706,-0.025431,-0.975473,...,1,-1.384400,0.910501,1.702404,"[0.06341613084077835, -0.09620142728090286, 0....","[0.3460847, -0.07331242, -0.35644135, 0.922641...",1.500148,-1.207502,0.483155,0.680081
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3163,19981610,29416526.0,0.129954,-2.602416,1,0,-0.354456,-0.116510,-0.087716,0.325979,...,1,-1.384400,-0.711784,-0.525417,"[0.030010048300027847, -0.1173621267080307, -0...","[0.15528555, -0.37537834, -0.03744465, 0.54269...",-1.779125,0.782860,0.649295,-1.360094
3164,19983009,26466419.0,-0.279725,-0.332271,0,1,-1.017390,0.005562,-0.025431,0.651342,...,3,0.839626,-0.404865,-0.120358,"[0.06512671709060669, -0.09365414828062057, -0...","[0.004019415, -0.086740956, 0.05278576, 0.7256...",1.504730,1.547878,-0.404655,-0.861766
3165,19990141,24852269.0,-0.095507,1.159539,0,0,-0.966395,-0.360654,-0.025431,0.651342,...,1,-1.384400,-1.062548,-0.525417,"[0.06681495159864426, -0.11060956865549088, -0...","[0.44183815, -0.14860332, -0.14596382, 0.75898...",0.774843,-1.685312,0.344599,1.542155
3166,19992875,29951097.0,-0.092758,-1.564635,0,0,-0.711421,-0.971014,-0.025431,0.651342,...,3,-0.272387,-0.361020,0.352210,"[0.05936222895979881, -0.12810805439949036, -0...","[0.5521446, 0.034858927, 0.0056764483, 0.53335...",-0.224904,0.007124,0.809153,-0.953249


In [104]:
from sklearn.ensemble import RandomForestClassifier

# Prepare numeric features
X = df_normalized[['BMI (kg/m2)', 'anchor_age', 'gender', 'stay', 'heartrate', 'temperature', 'resprate',
                         'o2sat', 'sbp', 'dbp', 'drg_code', 'drg_severity', 'drg_mortality', 
                         'Blood Pressure_h', 'Blood Pressure_l', 'chief_reduced', 'text_dim1', 'text_dim2', 'text_dim3']]

y = df_normalized['dod'].astype(int)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5230)

# Initialize the model
rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=4,
    random_state=5230,
    n_jobs=-1,
    max_features='sqrt' 
)

# Train the model
rf_model.fit(X_train, y_train)

# Predict
y_pred = rf_model.predict(X_test)
y_pred_prob = rf_model.predict_proba(X_test)[:, 1]

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("AU-ROC Score:", roc_auc_score(y_test, y_pred_prob))


Accuracy: 0.6829652996845426

Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.79      0.74       366
           1       0.65      0.54      0.59       268

    accuracy                           0.68       634
   macro avg       0.68      0.66      0.67       634
weighted avg       0.68      0.68      0.68       634

AU-ROC Score: 0.7374398499306745


In [132]:
from sklearn.decomposition import PCA

# Apply PCA to reduce 'chiefcomplaint_vector' to 1 dimension
pca = PCA(n_components=1)
df_normalized['chief_reduced'] = pca.fit_transform(np.vstack(df_normalized['chiefcomplaint_vector'].values))

# Apply PCA to reduce 'text_vector' to 3 dimensions
pca_text = PCA(n_components=3)
df_normalized[['text_dim1', 'text_dim2', 'text_dim3']] = pca_text.fit_transform(np.vstack(df_normalized['text_vector'].values))

# Perform z-score normalization on the 4 new vectors
df_normalized['chief_reduced'] = zscore(df_normalized['chief_reduced'])
df_normalized['text_dim1'] = zscore(df_normalized['text_dim1'])
df_normalized['text_dim2'] = zscore(df_normalized['text_dim2'])
df_normalized['text_dim3'] = zscore(df_normalized['text_dim3'])

In [146]:
from sklearn.ensemble import RandomForestClassifier

# Prepare numeric features
X = df_normalized[['BMI (kg/m2)', 'anchor_age', 'gender', 'stay', 'heartrate', 'temperature', 'resprate',
                         'o2sat', 'sbp', 'dbp', 'drg_code', 'drg_severity', 'drg_mortality', 
                         'Blood Pressure_h', 'Blood Pressure_l', 'chief_reduced', 'text_dim1', 'text_dim2', 'text_dim3']]

y = df_normalized['dod'].astype(int)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5230)

# Initialize the model
rf_model = RandomForestClassifier(
    n_estimators=120,
    max_depth=4,
    random_state=5230,
    n_jobs=-1,
    max_features='sqrt' 
)

# Train the model
rf_model.fit(X_train, y_train)

# Predict
y_pred = rf_model.predict(X_test)
y_pred_prob = rf_model.predict_proba(X_test)[:, 1]

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("AU-ROC Score:", roc_auc_score(y_test, y_pred_prob))


Accuracy: 0.6782334384858044

Classification Report:
               precision    recall  f1-score   support

           0       0.70      0.78      0.74       366
           1       0.64      0.53      0.58       268

    accuracy                           0.68       634
   macro avg       0.67      0.66      0.66       634
weighted avg       0.67      0.68      0.67       634

AU-ROC Score: 0.7382248593100074
